# 04 · Resilience & Percolation — European Air Transport Network

> **Data vintage — June 2014.** A frozen OpenFlights route snapshot, *not* the current network. Every threshold below describes 2014 connectivity. Percolation thresholds are topological invariants of the route graph, so a dated snapshot is the standard, defensible choice.

**This notebook answers Research Question 3: how vulnerable is the network to targeted hub attacks versus random failures?** It stages the classic **percolation experiment** of Albert, Jeong & Barabási (2000) on the undirected giant component:

1. Removes airports in three orders — **uniformly at random** (averaged over 20 runs), by **descending degree**, and by **descending betweenness** — tracking the giant component after every removal.
2. Plots the three decay curves on one axis; their divergence *is* the Barabási result — **error tolerance coupled with attack vulnerability**.
3. Locates the **critical threshold** f_c (the fraction removed at which the giant component falls below half its original size) for each strategy.
4. Reads the persisted hub ranking from `network.db` (NB02) as a cross-check on the attack order.
5. Closes with a plain-language reading: what the loss of Frankfurt, Heathrow or Amsterdam would mean for European connectivity.

Graph construction lives in `src/build_graph.py`; paths, palette and the PNG helper in `src/utils.py` (imported, not redefined). References: Barabási & Albert (1999) *Science* **286**; Albert, Jeong & Barabási (2000) *Nature* **406**.

In [ ]:
import sys
import sqlite3
import warnings
from pathlib import Path
from contextlib import closing

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

%load_ext autoreload
%autoreload 2


def _find_src() -> Path:
    """Locate the repo's src/ dir so imports work regardless of launch cwd."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "build_graph.py").exists():
            return p / "src"
    raise FileNotFoundError("Could not locate src/ - run this from inside the repo.")


sys.path.insert(0, str(_find_src()))

import build_graph as bg
import utils

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Experiment constants (no magic numbers below) -------------------------- #
SEED: int = 42                    # base seed for the random-failure runs
N_RUNS: int = 20                  # random removal orders averaged (matches NB02's null)
COLLAPSE_THRESHOLD: float = 0.5   # f_c = fraction removed when S(f) drops below this

# --- Presentation: one colour per strategy, ordered by how damaging it is --- #
C_RANDOM: str = utils.CATEGORICAL_PALETTE[3]   # #06d6a0 green - least damaging
C_DEGREE: str = utils.COLORS["node_low"]       # #ffd166 amber
C_BETW: str = utils.COLORS["node_high"]        # #ef476f red   - most damaging


def style_dark(fig: plt.Figure, ax: plt.Axes) -> None:
    """Apply the project's dark 'aviation at night' theme to a matplotlib axis."""
    fig.patch.set_facecolor(utils.COLORS["bg"])
    ax.set_facecolor(utils.COLORS["bg"])
    for spine in ax.spines.values():
        spine.set_color(utils.COLORS["coastline"])
    ax.tick_params(colors=utils.COLORS["text"])
    ax.xaxis.label.set_color(utils.COLORS["text"])
    ax.yaxis.label.set_color(utils.COLORS["text"])
    ax.title.set_color(utils.COLORS["text"])
    ax.set_axisbelow(True)
    ax.grid(True, color=utils.COLORS["coastline"], alpha=0.25, lw=0.5)

## 1 · The giant component — the object of the experiment

Percolation asks a connectivity question — *does a route still exist between two arbitrary airports?* — which is symmetric, so like NB02's small-world test and NB03's community detection it runs on the **undirected projection's giant component** (the 544-node connected core, 97% of nodes). Direction and the carrier-count weight are irrelevant here: a link either carries the connection or, once its endpoint is removed, does not. We measure network health by **S(f)** — the size of the largest connected component after a fraction *f* of nodes is removed, as a fraction of the intact giant. S(0) = 1; the network has *disintegrated* when S falls to a small fraction and the once-single core has shattered into many disconnected pieces.

In [ ]:
digraph, undirected, edges, airports_eu = bg.build_networks()

giant = undirected.subgraph(
    max(nx.connected_components(undirected), key=len)
).copy()
N = giant.number_of_nodes()

print(f"Undirected graph : {undirected.number_of_nodes()} nodes, "
      f"{undirected.number_of_edges()} edges")
print(f"Giant component  : {N} nodes, {giant.number_of_edges()} edges "
      f"({N / undirected.number_of_nodes():.1%} of nodes) - the attack surface")

## 2 · Two failure modes, three removal orders

**Random failure** models component wear-out or an untargeted disruption: nodes fail uniformly at random. A single draw is noisy, so we average **S(f)** over 20 independent random orders and carry the spread as a band. **Targeted attack** models an adversary — or a correlated cascade — that strikes the most important airports first, removed in **descending degree** (the busiest hubs) and, separately, **descending betweenness** (the structural bridges NB02 surfaced). The two orders need not agree: a high-degree low-cost base is an *endpoint*, while a lower-degree gateway can be the *bottleneck* whose loss severs a whole periphery.

**Ranking is computed once on the intact network** — the static attack of Albert–Jeong–Barabási (2000) — not recomputed after each removal. The adaptive variant that re-scores the damaged graph at every step (Holme et al. 2002) is strictly more destructive but answers a different question; the static order is the canonical scale-free-resilience experiment and the honest match to the cited references. Betweenness is computed **unweighted** on the giant, consistent with NB02 (the carrier-count weight is a service proxy, not a distance). Ties break on IATA code so every run is reproducible.

In [ ]:
def largest_cc_fraction(graph: nx.Graph, n_original: int) -> tuple[float, int]:
    """Largest-component size (as a fraction of n_original) and component count.

    Args:
        graph: The (partially dismantled) graph.
        n_original: Node count of the intact network, the denominator for S(f).

    Returns:
        Tuple ``(S, n_components)``; ``(0.0, 0)`` for the empty graph.
    """
    if graph.number_of_nodes() == 0:
        return 0.0, 0
    components = list(nx.connected_components(graph))
    return max(len(c) for c in components) / n_original, len(components)


def percolate(
    graph: nx.Graph, removal_order: list[str]
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Remove nodes in the given order, tracking giant-component decay.

    Args:
        graph: Undirected graph to attack (copied internally, not mutated).
        removal_order: Nodes to remove, in order.

    Returns:
        Three arrays of length ``len(removal_order) + 1`` (index 0 = intact
        network): fraction removed, S(f), and the number of components.
    """
    n0 = graph.number_of_nodes()
    g = graph.copy()
    frac, s_frac, n_comp = [0.0], [1.0], [nx.number_connected_components(g)]
    for i, node in enumerate(removal_order, start=1):
        g.remove_node(node)
        s, c = largest_cc_fraction(g, n0)
        frac.append(i / n0)
        s_frac.append(s)
        n_comp.append(c)
    return np.array(frac), np.array(s_frac), np.array(n_comp)


def random_percolation(
    graph: nx.Graph, n_runs: int, seed: int
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Average percolation over ``n_runs`` uniformly-random removal orders.

    Args:
        graph: Undirected graph to attack.
        n_runs: Independent random orders to average.
        seed: Base seed for the reproducible RNG.

    Returns:
        Tuple ``(frac, s_mean, s_std, c_mean)`` on the shared fraction axis: the
        mean and standard deviation of S(f) across runs, and the mean component
        count.
    """
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    frac, s_runs, c_runs = None, [], []
    for _ in range(n_runs):
        order = [nodes[i] for i in rng.permutation(len(nodes))]
        frac, s, c = percolate(graph, order)
        s_runs.append(s)
        c_runs.append(c)
    s_runs, c_runs = np.array(s_runs), np.array(c_runs)
    return frac, s_runs.mean(axis=0), s_runs.std(axis=0), c_runs.mean(axis=0)


def critical_fraction(
    frac: np.ndarray, s_frac: np.ndarray, threshold: float = COLLAPSE_THRESHOLD
) -> float | None:
    """Fraction removed at which the giant component first drops below threshold.

    Linearly interpolates between the two bracketing steps for a sub-step estimate.

    Args:
        frac: Ascending fraction-removed axis.
        s_frac: Giant-component fraction at each step.
        threshold: Collapse threshold (default 0.5 of the intact giant).

    Returns:
        The interpolated crossing fraction, or ``None`` if S never drops below it.
    """
    below = np.flatnonzero(s_frac < threshold)
    if below.size == 0:
        return None
    i = int(below[0])
    if i == 0:
        return float(frac[0])
    x0, x1, y0, y1 = frac[i - 1], frac[i], s_frac[i - 1], s_frac[i]
    return float(x0 + (threshold - y0) * (x1 - x0) / (y1 - y0))


# Static rankings on the intact giant (computed once).
degree = dict(giant.degree())
betweenness = nx.betweenness_centrality(giant, normalized=True)


def ranked_order(score: dict[str, float]) -> list[str]:
    """Nodes by descending score, ties broken by IATA for reproducibility."""
    return sorted(giant.nodes(), key=lambda n: (score[n], n), reverse=True)


degree_order = ranked_order(degree)
betweenness_order = ranked_order(betweenness)

print("First five removed by each targeted order:")
print(f"  degree      : {degree_order[:5]}")
print(f"  betweenness : {betweenness_order[:5]}")

### Cross-check — the hubs we strike first (from `network.db`)

NB02 persisted every centrality to `network.db`. Before attacking, we read the top airports by betweenness straight from SQL — the order a betweenness attack removes them in — and confirm the stored (directed-graph) betweenness ranks the giant the same way as the undirected-giant betweenness driving the attack. The query is saved as **`sql/queries/hub_ranking.sql`**. The read is a cross-check only; the attack computes its own ordering above, so the notebook still runs end-to-end on a fresh database.

In [ ]:
query_path = utils.SQL_QUERIES_DIR / "hub_ranking.sql"
sql = query_path.read_text(encoding="utf-8")
print(sql)

hub_table = None
try:
    with closing(sqlite3.connect(utils.DB_PATH)) as conn:
        hub_table = pd.read_sql(sql, conn)
except Exception as exc:  # noqa: BLE001 - cross-check is optional, attack is not
    print(f"DB read skipped ({type(exc).__name__}): run load_db.py then NB02 first.")

if hub_table is not None and hub_table["betweenness"].notna().any():
    with closing(sqlite3.connect(utils.DB_PATH)) as conn:
        persisted = (
            pd.read_sql(
                "SELECT iata, betweenness FROM nodes WHERE betweenness IS NOT NULL",
                conn,
            )
            .set_index("iata")["betweenness"]
        )
    common = [n for n in giant.nodes() if n in persisted.index]
    rho, _ = spearmanr(
        persisted.loc[common].to_numpy(),
        np.array([betweenness[n] for n in common]),
    )
    print(f"Spearman rho, persisted (directed) vs undirected-giant betweenness: "
          f"{rho:.3f}  -> the SQL ranking previews the attack's hit order.")
    display(hub_table)
else:
    print("betweenness column is NULL - SQL cross-check skipped (attack is unaffected).")

## 3 · Running the three strategies

Each removal order is played to completion, recording after every node the fraction removed, S(f) and the number of connected components. Random is averaged over the 20 seeded runs; the two targeted orders are deterministic. All three share the same fraction axis — one node removed per step.

In [ ]:
frac_r, s_r_mean, s_r_std, c_r_mean = random_percolation(giant, N_RUNS, SEED)
frac_d, s_d, c_d = percolate(giant, degree_order)
frac_b, s_b, c_b = percolate(giant, betweenness_order)
frac = frac_d  # identical axis for all three strategies


def s_at(frac_axis: np.ndarray, s_curve: np.ndarray, f: float) -> float:
    """Giant-component fraction at the removal step closest to fraction ``f``."""
    return float(s_curve[np.argmin(np.abs(frac_axis - f))])


print("Giant component S(f) after removing 10% / 20% of airports:")
print(f"  random failure        : {s_at(frac, s_r_mean, .10):.2f} / {s_at(frac, s_r_mean, .20):.2f}")
print(f"  targeted (degree)     : {s_at(frac, s_d, .10):.2f} / {s_at(frac, s_d, .20):.2f}")
print(f"  targeted (betweenness): {s_at(frac, s_b, .10):.2f} / {s_at(frac, s_b, .20):.2f}")

## 4 · The critical threshold f_c

The **percolation threshold** f_c is the fraction of nodes whose removal collapses the giant component — operationally, where S(f) first falls below **half** its original size. For a **broad-scale / heavy-tailed** network like this one (NB02: the degree distribution is a truncated power law, not a clean scale-free), theory predicts a wide gap between the two failure modes — robustness to random loss, because a random node is almost surely a low-degree redundant spoke, but fragility to attack, because a handful of hubs and bridges hold the network together. f_c is interpolated between the two bracketing removal steps.

In [ ]:
fc_random = critical_fraction(frac_r, s_r_mean)
fc_degree = critical_fraction(frac_d, s_d)
fc_betw = critical_fraction(frac_b, s_b)

fc_table = pd.DataFrame([
    {"strategy": "Random failure (mean of 20)", "f_c (giant < 50%)": fc_random,
     "S(0.10)": s_at(frac, s_r_mean, .10), "S(0.20)": s_at(frac, s_r_mean, .20)},
    {"strategy": "Targeted - degree", "f_c (giant < 50%)": fc_degree,
     "S(0.10)": s_at(frac, s_d, .10), "S(0.20)": s_at(frac, s_d, .20)},
    {"strategy": "Targeted - betweenness", "f_c (giant < 50%)": fc_betw,
     "S(0.10)": s_at(frac, s_b, .10), "S(0.20)": s_at(frac, s_b, .20)},
]).round(3)

gap = fc_random / fc_betw
print(f"Robustness gap: random failure tolerates {gap:.1f}x the removal that the "
      f"betweenness attack needs to break the network.\n")
fc_table

## 5 · The resilience curve

The headline figure. Three decay curves on one axis, with vertical drop-lines marking each strategy's f_c against the 50% threshold. The **divergence between the random and targeted curves is the Barabási (2000) result** — the network shrugs off random loss but comes apart under a targeted one. Exported to `figures/resilience_curve.png` for the README banner.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

# Random failure: mean curve + ±1 sd band across the 20 runs.
ax.fill_between(frac_r, s_r_mean - s_r_std, s_r_mean + s_r_std,
                color=C_RANDOM, alpha=0.18, linewidth=0)
ax.plot(frac_r, s_r_mean, color=C_RANDOM, lw=2.2,
        label=f"Random failure (20 runs)      f_c = {fc_random:.2f}")
ax.plot(frac_d, s_d, color=C_DEGREE, lw=2.2,
        label=f"Targeted \u2014 degree             f_c = {fc_degree:.2f}")
ax.plot(frac_b, s_b, color=C_BETW, lw=2.2,
        label=f"Targeted \u2014 betweenness        f_c = {fc_betw:.2f}")

# 50% collapse threshold and the f_c drop-lines.
ax.axhline(COLLAPSE_THRESHOLD, color=utils.COLORS["text"], ls=":", lw=1, alpha=0.55)
ax.text(0.99, COLLAPSE_THRESHOLD + 0.015, "50% of giant", ha="right", va="bottom",
        color=utils.COLORS["text"], fontsize=8, alpha=0.75)
for fc_val, col in [(fc_betw, C_BETW), (fc_degree, C_DEGREE), (fc_random, C_RANDOM)]:
    if fc_val is not None:
        ax.axvline(fc_val, color=col, ls="--", lw=1, alpha=0.5, ymax=0.5)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Fraction of airports removed,  f")
ax.set_ylabel("Giant component  S(f) / S(0)")
ax.set_title("Network resilience: random failure vs targeted attack\n"
             f"European air transport, {utils.DATA_VINTAGE}")
style_dark(fig, ax)
ax.legend(facecolor=utils.COLORS["land"], edgecolor=utils.COLORS["coastline"],
          labelcolor=utils.COLORS["text"], fontsize=9, loc="upper right")

utils.save_fig_png(fig, "resilience_curve")
plt.show()

## 6 · Fragmentation — how the core shatters

S(f) tracks the largest surviving piece; the **number of connected components** tracks how the rest breaks up. Under a targeted attack the giant does not shrink smoothly — it *shatters*, spawning a burst of small disconnected fragments. The **peak in the fragment count locates the percolation transition**: the point where the last large cluster has finally broken apart. Random removal produces far fewer fragments, and only at much higher *f*.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(frac_r, c_r_mean, color=C_RANDOM, lw=2.2, label="Random failure (20 runs)")
ax.plot(frac_d, c_d, color=C_DEGREE, lw=2.2, label="Targeted \u2014 degree")
ax.plot(frac_b, c_b, color=C_BETW, lw=2.2, label="Targeted \u2014 betweenness")

# Mark the fragmentation peak of the betweenness attack (the percolation signature).
peak_i = int(np.argmax(c_b))
ax.scatter([frac_b[peak_i]], [c_b[peak_i]], color=C_BETW, s=40, zorder=5)
ax.annotate(f"shatters into {int(c_b[peak_i])} pieces\nat f = {frac_b[peak_i]:.2f}",
            xy=(frac_b[peak_i], c_b[peak_i]),
            xytext=(frac_b[peak_i] - 0.35, c_b[peak_i] - 30),
            color=utils.COLORS["text"], fontsize=8,
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=-0.2", color=utils.COLORS["coastline"], lw=1))

ax.set_xlim(0, 1)
ax.set_xlabel("Fraction of airports removed,  f")
ax.set_ylabel("Number of connected components")
ax.set_title("Fragmentation: how the network breaks apart\n"
             f"European air transport, {utils.DATA_VINTAGE}")
style_dark(fig, ax)
ax.legend(facecolor=utils.COLORS["land"], edgecolor=utils.COLORS["coastline"],
          labelcolor=utils.COLORS["text"], fontsize=9, loc="upper right")

utils.save_fig_png(fig, "fragmentation")
plt.show()

## 7 · Reading Research Question 3

**European aviation is robust to random failure but fragile to targeted attack — the defining signature of a heavy-tailed network.** Removing airports at random, the giant component holds above half its size until roughly **44%** are gone (f_c ≈ 0.44); striking the busiest hubs by degree collapses it at **~18%** (f_c ≈ 0.18), and striking the structural bridges by betweenness collapses it fastest of all, at **~15%** (f_c ≈ 0.15) — the network tolerates nearly **three times** the damage when the loss is random rather than aimed. That betweenness beats degree repeats NB02's lesson: the airports holding the network together are the **bridges** (Istanbul, the Nordic and Greek-island gateways), not simply the largest low-cost bases — a super-hub is an endpoint, a gateway is a bottleneck.

In plain terms: losing any single airport to weather or a strike barely dents European connectivity, because almost every airport is a redundant spoke. But a disruption that took out a small set of the top hubs and gateways together — an **Amsterdam** or a **Frankfurt** among them — would not degrade the network gracefully; it would fracture it into regional islands, cutting off the thinly-connected periphery (the Norwegian coast, the Greek islands, the Anatolian interior) whose only link to the rest ran through those bridges. This is why hub redundancy and gateway diversity — not raw capacity — are the levers that matter for aviation resilience. **Two caveats, both from earlier notebooks:** the numbers are June 2014, and **Heathrow** ranks low here only because the codeshare filter and its slot constraints understate it (NB02) — its real-world criticality is far higher than this operating-carrier snapshot shows.

## Summary

**What we built.** The project's percolation centrepiece: on the 544-airport giant component, node removal under three orders — random (20 seeded runs), descending degree, descending betweenness — tracking **S(f)** and the fragment count after every removal, the critical threshold f_c extracted per strategy, a SQL cross-check of the attack order against NB02's persisted centralities, and two figures (the resilience curve, exported for the README, and the fragmentation view).

**What stood out.** **Robust yet fragile — the Barabási (2000) result, confirmed on real infrastructure.** Random failure tolerates ~44% node loss before the giant halves; a degree attack needs ~18% and a betweenness attack just ~15% — nearly a **3× robustness gap** — and the bridges (betweenness) matter more than the biggest hubs (degree), echoing RQ1. The heavy-tailed degree distribution from NB02 is exactly what produces this asymmetry: many redundant spokes, a few load-bearing hubs.

**Next — README & the Quarto site.** All four notebooks are complete. What remains is the write-up: the three findings with numbers, the visual gallery (route map, community map, resilience curve), the four-command reproduce block, and rendering the notebooks into the linked GitHub Pages site.